# CAVE pre/post V1DD addition

Fold 0, K=10, no confidence exclusion. Compare mean, pointwise MLP and GT across:
- MICrONS + V1DD training (`with_v1dd`) only.
- `pre_only`, `pre_post`, and matched-width `pre_mean_control`.

All conditions use the same eligible post sites and paired held-out cells. Site
eligibility excludes unresolved roots and empty post masks before presynaptic
window deduplication. The excluded invalid V1DD root is not used.

V1DD post-site inventory: 1,037,698 sites across 265 cells, 254 spatial shards;
27 unresolved roots. Roots and coordinates come from outgoing synapses_v1dd at
materialization 1196. Exact 129³ crops are masked by the postsynaptic root at that
timestamp; SegCLR uses FP16, torch.compile, fixed/padded batch 128 and the existing
compilation cache. CPU workers share compressed packed image/root-label chunks with retained ownership. GPUs consume only completed prepared inputs.
MICrONS post-site vectors are reused from the verified v1718 inference.

The prepared post cache retains synapse IDs, post coordinates (nm), post root IDs,
eligibility and 64D embeddings, aligned to the CAVE window rows.

Job chain:
- Site preparation: 24562166 (completed).
- Shared CPU preparation: 24565943 (64 normal), 24565944 (128 preemptable); GPU dispatcher: 24565945.
- Verify inference: 24562337.
- Join post cache, manifests and loader checks: 24562381.
- Training: 24562383 (4 models preemptable), 24562394 (5 normal GPU).
- Paired domain summaries: 24562395.

Database: `/orcd/scratch/orcd/013/jcbliao/presynaptic_axons/cave_pre_post_v1dd_addition`.
Results: `results/presynaptic/cave_pre_post_v1dd_addition`.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/train_gnn.py').exists())
root = repo / 'results/presynaptic/cave_pre_post_v1dd_addition'
analysis = repo / 'analysis/presynaptic/cave_pre_post_v1dd_addition'


In [ ]:
# Completed models; rerun while jobs progress.
rows = []
for path in root.glob('with_v1dd/*/gnn_*.json'):
    r = json.loads(path.read_text())
    domains = {'combined': {'cell': r['test_metrics'], 'window': r['window_test_metrics']}, **r['dataset_test_metrics']}
    for domain, metrics in domains.items():
        rows.append(dict(augmentation=path.parent.parent.name, feature=path.parent.name, architecture=r['args']['architecture'], dataset=domain, best_epoch=r['best_epoch'], window_macro_f1=metrics['window']['macro_f1'], cell_macro_f1=metrics['cell']['macro_f1']))
results = pd.DataFrame(rows)
display(results if len(results) else pd.DataFrame({'status': ['Models are queued or running']}))


In [ ]:
if len(results):
    for domain in results.dataset.unique():
        table = results[results.dataset.eq(domain)].pivot(index=['augmentation', 'architecture'], columns='feature', values='window_macro_f1')
        display(table)
        table.plot.bar(title=f'{domain}: window macro F1', ylim=(0, 1), figsize=(10, 4))
        plt.tight_layout()


In [ ]:
for name in ['post_deltas.csv']:
    path = analysis / name
    if path.exists():
        print(name)
        display(pd.read_csv(path))


In [ ]:
database = Path('/orcd/scratch/orcd/013/jcbliao/presynaptic_axons/cave_pre_post_v1dd_addition')
path = database / 'postsynaptic_inventory.csv'
if path.exists():
    inventory = pd.read_csv(path, dtype={'root_id': str})
    display(inventory.groupby('source_dataset')[['n_valid_pre', 'n_valid_pre_post']].sum())
    display(inventory[inventory.n_valid_pre_post.eq(0)])
